# Agentic YouTube Video Assistant (RAG)

This notebook walks through the pipeline step by step, for studying/explaining
in an interview. The actual app (FastAPI + Streamlit) imports the same logic
from `src/` so this notebook and the running app never diverge — this notebook
is just the annotated, cell-by-cell version of it.

**Pipeline:** transcript → chunks → embeddings → FAISS → retriever → LCEL chain → LLM → (agent routes to retrieval / summary / web search)


In [ ]:
import sys
sys.path.insert(0, "..") if "src" not in __import__("os").listdir(".") else None

from dotenv import load_dotenv
load_dotenv()


## Step 1a — Indexing: fetch the transcript

`YouTubeTranscriptApi` pulls the auto-generated or uploaded captions for a
video id. We accept either a full URL or a bare id — `extract_video_id`
handles both, which is what makes this "seamless" to use rather than
requiring the user to know the raw 11-character id.


In [ ]:
from src.ingestion import extract_video_id, fetch_transcript, split_transcript

# Swap this for any YouTube video with captions enabled
VIDEO_URL = "https://www.youtube.com/watch?v=BV7QRw8wtjs"

video_id = extract_video_id(VIDEO_URL)
transcript = fetch_transcript(video_id)
print(f"video_id = {video_id}")
print(f"transcript length = {len(transcript)} chars")
print(transcript[:500], "...")


## Step 1b — Indexing: split into chunks

`RecursiveCharacterTextSplitter` tries to split on paragraph, then sentence,
then word boundaries before resorting to a hard character cut — so chunks stay
semantically coherent instead of being sliced mid-sentence. `chunk_overlap=200`
means consecutive chunks share 200 characters, so an answer that straddles a
chunk boundary doesn't lose context on either side.


In [ ]:
chunks = split_transcript(transcript)
print(f"{len(chunks)} chunks")
chunks[0]


## Step 2 — Indexing: embeddings + FAISS vector store

Each chunk gets turned into a dense vector (384 dimensions, from the
`all-MiniLM-L6-v2` sentence-transformer model — small, fast, runs locally for
free, no API cost). FAISS then indexes those vectors for fast approximate
nearest-neighbor search: given a query vector, which chunk vectors are
closest to it?


In [ ]:
from src.vectorstore import build_vector_store

vector_store = build_vector_store(chunks)
print(f"{vector_store.index.ntotal} vectors indexed")


## Step 3 — Retrieval: similarity vs MMR

Two retrieval strategies:

- **similarity**: return the k chunks closest to the query embedding. Simple,
  but if the top matches are near-duplicates of each other (common in a video
  that repeats a point), you get redundant context instead of broad coverage.
- **MMR (Maximal Marginal Relevance)**: pulls a larger candidate pool
  (`fetch_k`), then greedily selects k of them balancing relevance to the
  query against how different they are from chunks already picked
  (`lambda_mult` tunes that balance — 1.0 = pure relevance, 0.0 = pure
  diversity).

> Note: the original version of this notebook had a bug here —
> `seach_kwargs={"k", 4}` was misspelled (should be `search_kwargs`) **and**
> `{"k", 4}` was a Python *set* literal, not a dict (`{"k": 4}`). LangChain
> silently ignored the unrecognized kwarg rather than erroring, and FAISS's
> own default of k=4 happened to mask it — so it looked like it worked but
> the setting was never actually applied.


In [ ]:
from src.retrieval import get_similarity_retriever, get_mmr_retriever

query = "How to make the mind bulletproof"

similarity_retriever = get_similarity_retriever(vector_store, k=4)
mmr_retriever = get_mmr_retriever(vector_store, k=4, fetch_k=20, lambda_mult=0.5)

similarity_docs = similarity_retriever.invoke(query)
mmr_docs = mmr_retriever.invoke(query)

print("--- similarity ---")
for d in similarity_docs:
    print(d.page_content[:80].replace("\n", " "), "...")

print("\n--- mmr ---")
for d in mmr_docs:
    print(d.page_content[:80].replace("\n", " "), "...")


## Step 4 — Augmentation + Generation: the LCEL chain

`RunnableParallel` runs the retrieval branch (`retriever | format_docs`) and
the question passthrough (`RunnablePassthrough`) *concurrently*, merging their
results into `{"context": ..., "question": ...}` — exactly the input shape the
prompt template expects. That merged dict flows into the prompt, then the
LLM, then an output parser. The whole thing is one composed `Runnable`, which
is what "declarative LCEL chain" means: you're describing *what* the pipeline
is, not writing imperative glue code to wire each step by hand.


In [ ]:
from src.chains import build_rag_chain

rag_chain = build_rag_chain(similarity_retriever)
answer = rag_chain.invoke("What is this video about?")
print(answer)


## Structured output — mitigating hallucinations

A plain `StrOutputParser` just returns a string — the model can hedge or be
subtly wrong and nothing in the code can act on that. Binding the LLM to a
Pydantic schema via `.with_structured_output()` forces OpenAI's function-
calling machinery to return fields matching `VideoAnswer` exactly: `answer`,
`key_topics`, `is_grounded`, `confidence`. The app can then programmatically
check `is_grounded` / `confidence` before trusting an answer, instead of
blindly trusting prose.


In [ ]:
from src.chains import build_structured_rag_chain

structured_chain = build_structured_rag_chain(similarity_retriever)
structured_answer = structured_chain.invoke("What is this video about?")
structured_answer


## Full-video summarization

Summarization intentionally skips retrieval — retrieval finds the top-k
chunks *relevant to a question*, but a summary needs the *whole* video, so we
stuff the entire transcript into the prompt instead.


In [ ]:
from src.chains import build_summary_chain

summary_chain = build_summary_chain()
print(summary_chain.invoke({"context": transcript}))


## The agent — dynamic routing between tools

Everything above is a *fixed* pipeline: retrieve, then answer. The agent
instead gives the LLM three tools and lets it decide, per query, what to call:

1. `retrieve_from_video` — grounded vector retrieval over this video's transcript
2. `summarize_video` — full-video summary
3. `web_search` — DuckDuckGo search, for anything outside the transcript's scope

`create_agent` (LangChain 1.x, built on LangGraph) wires this up as a
compiled graph: run the LLM -> if it emitted a tool call, execute it and feed
the result back in as a message -> repeat until the LLM returns a final
answer instead of another tool call. You invoke it with `{"messages": [...]}`
and the answer is the last message's `.content`.


In [ ]:
from src.session import VideoSession
from src.agent import build_agent_executor, run_agent

session = VideoSession(video_id=video_id, chunks=chunks, vector_store=vector_store)
agent = build_agent_executor(session)

# A question the transcript can answer -> should route to retrieve_from_video
print(run_agent(agent, "What did the guest say about pain during the race?"))


In [ ]:
# A question the transcript can't answer -> should route to web_search
print(run_agent(agent, "What is the current world record for the Backyard Ultra format in general?"))


## Where this goes next

`src/` is imported directly by `main.py` (FastAPI) and `streamlit_app.py` —
run those for the actual app instead of re-running this notebook each time.
See `README.md` for the full architecture diagram and run instructions.
